# Weather–Subway Data Merger

기상 베이스 데이터와 지하철 혼잡도 데이터를 시간 단위로 병합하여 모델 입력용 데이터를 생성한다.

- 학습 데이터: 2021–2023년
- 검증 데이터: 2024년
- 병합 기준: Datetime


병합용 기상 베이스에 지하철 혼잡도(inner/outer)를 Datetime 기준으로 매칭  
이후 모델 입력용 파생 변수(prev_1h, prev_24h)를 생성하여 processed CSV 저장

※ 실제 베이스 CSV 및 원본 데이터는 GitHub에 포함하지 않는다.

In [ ]:
import os
import zipfile
import pandas as pd

# 0) 환경 설정 (경로/파일명)
BASE_DIR = "."  
RAW_DIR = os.path.join(BASE_DIR, "data", "raw")          # 원본/중간 파일 위치
OUT_DIR = os.path.join(BASE_DIR, "data", "processed")    # 결과 저장 폴더
os.makedirs(OUT_DIR, exist_ok=True)

OPTIONAL_ZIP_PATH = os.path.join(RAW_DIR, "2024_역별_승하차_혼잡도.zip")
OPTIONAL_EXTRACT_DIR = os.path.join(RAW_DIR, "역별_승하차_혼잡도_해제")

BASE_WEATHER_2024 = os.path.join(RAW_DIR, "2024_병합용_데이터.csv")
BASE_WEATHER_2021_2023 = os.path.join(RAW_DIR, "2021_3_병합용_데이터.csv")

# 역별 지하철 파일 디렉토리(해제 후 폴더 또는 원래 폴더)
SUBWAY_DIR = OPTIONAL_EXTRACT_DIR  

# 출력 파일명
OUT_PREFIX = "processed_"  # processed_{역명}.csv 형태

PREV_1H_SHIFT = 2
PREV_24H_SHIFT = 48

# 1) zip 해제
def extract_zip_if_exists(zip_path: str, extract_dir: str) -> None:
    """zip이 존재하면 해제 (옵션 기능)"""
    if not os.path.exists(zip_path):
        print(f"[옵션] zip 없음 → 스킵: {zip_path}")
        return
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)
    print(f"[옵션] zip 해제 완료: {extract_dir}")

# 2) 로드 함수
def load_base_weather(path: str) -> pd.DataFrame:
    """병합용 기상 베이스 로드 (Datetime 필수)"""
    if not os.path.exists(path):
        raise FileNotFoundError(f"베이스 파일을 찾을 수 없음: {path}")
    df = pd.read_csv(path)
    if "Datetime" not in df.columns:
        raise ValueError("베이스 파일에 Datetime 컬럼이 필요합니다.")
    df["Datetime"] = pd.to_datetime(df["Datetime"])
    return df

def load_subway_single(path: str) -> pd.DataFrame:
    """
    역별 지하철 파일 로드
    기대 컬럼(네 기존 코드 기준):
      - 날짜, hour, 호선, inner_congestion_pct, outer_congestion_pct
    """
    df = pd.read_csv(path)

    required = {"날짜", "hour", "호선", "inner_congestion_pct", "outer_congestion_pct"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"지하철 파일 컬럼 누락({os.path.basename(path)}): {missing}")

    out = df[["날짜", "hour", "호선", "inner_congestion_pct", "outer_congestion_pct"]].copy()
    out["Datetime"] = pd.to_datetime(out["날짜"].astype(str) + " " + out["hour"].astype(str) + ":00")
    out = out.rename(columns={"호선": "Line"})
    return out[["Datetime", "Line", "inner_congestion_pct", "outer_congestion_pct"]]

# 3) 병합 + lag 생성
def merge_subway_into_base(df_base: pd.DataFrame, df_sub: pd.DataFrame) -> pd.DataFrame:
    """Datetime 기준 병합 후, NaN만 채우는 방식으로 정리"""
    df_base = df_base.copy()
    df_sub = df_sub.copy()

    df_merged = pd.merge(
        df_base,
        df_sub,
        on="Datetime",
        how="left",
        suffixes=("", "_filled"),
    )

    # NaN만 filled값으로 채우기
    for col in ["Line", "inner_congestion_pct", "outer_congestion_pct"]:
        fill_col = f"{col}_filled"
        if fill_col in df_merged.columns:
            if col not in df_merged.columns:
                df_merged[col] = pd.NA
            df_merged[col] = df_merged[col].where(~df_merged[col].isna(), df_merged[fill_col])
            df_merged.drop(columns=[fill_col], inplace=True)

    return df_merged

def add_lag_features(df: pd.DataFrame,
                     prev_1h_shift: int = PREV_1H_SHIFT,
                     prev_24h_shift: int = PREV_24H_SHIFT) -> pd.DataFrame:
    """이전 혼잡도 파생변수 생성 + 시작부 결측을 '순환' 방식으로 보정"""
    df = df.sort_values("Datetime").copy()

    df["prev_1h_inner_congestion_pct"] = df["inner_congestion_pct"].shift(prev_1h_shift)
    df["prev_1h_outer_congestion_pct"] = df["outer_congestion_pct"].shift(prev_1h_shift)
    df["prev_24h_inner_congestion_pct"] = df["inner_congestion_pct"].shift(prev_24h_shift)
    df["prev_24h_outer_congestion_pct"] = df["outer_congestion_pct"].shift(prev_24h_shift)

    # (1) prev_1h 시작부 결측: 마지막 쪽 값을 가져와 순환 보정
    if df["inner_congestion_pct"].notna().any():
        df.loc[df["prev_1h_inner_congestion_pct"].isna(), "prev_1h_inner_congestion_pct"] = df["inner_congestion_pct"].iloc[-prev_1h_shift]
        df.loc[df["prev_1h_outer_congestion_pct"].isna(), "prev_1h_outer_congestion_pct"] = df["outer_congestion_pct"].iloc[-prev_1h_shift]

    # (2) prev_24h 시작부 결측: 마지막 24h 구간을 앞쪽에 채움
    total_len = len(df)
    if total_len >= prev_24h_shift:
        df.iloc[0:prev_24h_shift, df.columns.get_loc("prev_24h_inner_congestion_pct")] = \
            df.iloc[total_len-prev_24h_shift:total_len, df.columns.get_loc("inner_congestion_pct")].values
        df.iloc[0:prev_24h_shift, df.columns.get_loc("prev_24h_outer_congestion_pct")] = \
            df.iloc[total_len-prev_24h_shift:total_len, df.columns.get_loc("outer_congestion_pct")].values

    return df

# 4) 역별 일괄 처리
def infer_station_name(filename: str) -> str:
    """파일명에서 역명 추출: '{역명}_내외선_승하차_혼잡도.csv'"""
    base = os.path.basename(filename)
    return base.replace("_내외선_승하차_혼잡도.csv", "").strip()

def build_processed_files(base_weather_path: str,
                          subway_dir: str,
                          out_dir: str) -> None:
    """역별 파일들을 읽어 병합 후 processed_{역명}.csv 저장"""
    df_base = load_base_weather(base_weather_path)

    if not os.path.isdir(subway_dir):
        raise FileNotFoundError(f"지하철 파일 폴더가 없습니다: {subway_dir}")

    files = [f for f in os.listdir(subway_dir) if f.endswith("_내외선_승하차_혼잡도.csv")]
    if not files:
        raise FileNotFoundError(f"지하철 입력 파일이 없습니다: {subway_dir}")

    print(f"처리 대상 역 파일 수: {len(files)}개")

    for fname in files:
        in_path = os.path.join(subway_dir, fname)
        station = infer_station_name(fname)
        out_path = os.path.join(out_dir, f"{OUT_PREFIX}{station}.csv")

        try:
            df_sub = load_subway_single(in_path)
            merged = merge_subway_into_base(df_base, df_sub)
            merged = add_lag_features(merged)

            merged.to_csv(out_path, index=False, encoding="utf-8-sig")
            print(f"저장 완료: {out_path}")
        except Exception as e:
            print(f"실패({station}): {e}")

# 5) 실행 (원하는 연도 베이스 선택)
# zip이 있으면 먼저 해제
extract_zip_if_exists(OPTIONAL_ZIP_PATH, OPTIONAL_EXTRACT_DIR)

# 2024 검증 데이터 기준으로 processed 생성
build_processed_files(
    base_weather_path=BASE_WEATHER_2024,
    subway_dir=SUBWAY_DIR,
    out_dir=OUT_DIR
)

print("전체 처리 종료")